## Extraction of Data from Chirps Dataset (GEE) ##

In [1]:
# Import Google Earth Engine API and Initialize it. 
import ee
import pandas as pd

ee.Authenticate()
ee.Initialize(project="ey-data-and-ai-challenge")

In [2]:
# Read coordinates and date from water quality training dataset, drop given features.

wq_df = pd.read_csv('../data/water_quality_training_dataset.csv')
wq_df = wq_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
wq_df['id'] = wq_df.index
wq_df['Sample Date'] = pd.to_datetime(wq_df['Sample Date'], format="%d-%m-%Y").dt.strftime("%Y-%m-%d")
wq_df.head()

,Latitude,Longitude,Sample Date,id
0,-28.760833,17.730278,2011-01-02,0
1,-26.861111,28.884722,2011-01-03,1
2,-26.450000,28.085833,2011-01-03,2
3,-27.671111,27.236944,2011-01-03,3
4,-27.356667,27.286389,2011-01-03,4


In [9]:
# Convert Coordinates and given date to ee.Features for use in batch export.

features = []

for index, row in wq_df.iterrows():
    feat = ee.Feature(
        ee.Geometry.Point([row['Longitude'], row['Latitude']]).buffer(5500), #add a 5.5km buffer in case of inexact coordinates
        {'id': row['id'],
         'start_date': (pd.to_datetime(row['Sample Date']) - pd.Timedelta(weeks=1)).strftime('%Y-%m-%d'),
         'end_date': (pd.to_datetime(row['Sample Date']) + pd.Timedelta(weeks=1)).strftime('%Y-%m-%d')
        }
    )
    features.append(feat)

fc = ee.FeatureCollection(features)         # create feature collection with features

In [10]:
chirps_collection = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select('precipitation')

In [14]:
def extract_median_values(feat):
    collection = chirps_collection.filterDate(feat.get('start_date'), feat.get('end_date'))
    img = collection.reduce(ee.Reducer.sum()) # reduce image collection into a single image

    chirps_col = img.reduceRegions(collection=ee.FeatureCollection([feat]), reducer=ee.Reducer.first(), scale = 5566)
    
    return chirps_col.first()

In [15]:
fc_mapped = fc.map(extract_median_values)

In [16]:
# Process data and export to Google Drive

task = ee.batch.Export.table.toDrive(
    collection=fc_mapped,
    description="chirps_csv_export",
    fileNamePrefix= "chirps_features_training",
    fileFormat='CSV'
)
task.start()